# ZIP-RC: Zero-Overhead Introspection for Adaptive Test-Time Compute

This notebook runs the complete ZIP-RC pipeline from start to finish. ZIP-RC enables LLMs to predict their own success probability (reward) and computational cost (remaining generation length) during inference—without adding any extra overhead.

**Pipeline Overview:**
1. **Setup** - Install dependencies and authenticate with HuggingFace
2. **Generate Rollouts** - Generate completions from the base model
3. **Score Rollouts** - Evaluate each completion for correctness
4. **Build Dataset** - Create prefix training examples with joint labels
5. **Train ZIP-RC** - Fine-tune the model to predict (reward, length) at each step
6. **Test Inference** - Verify the trained model can introspect

---

## 1. Setup & Environment Check

First, let's install the required dependencies and check our GPU availability.

In [ ]:
# Install required packages
!pip install -q transformers accelerate torch datasets huggingface_hub
print("Packages installed!")

In [ ]:
# Check GPU availability and memory
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU available: {gpu_name}")
    print(f"   Total memory: {gpu_memory:.1f} GB")
    
    # Memory warning for Llama-3.1-8B
    if gpu_memory < 16:
        print("\nWARNING: Llama-3.1-8B requires ~16GB VRAM in float16.")
        print("   You may encounter OOM errors. Consider:")
        print("   - Using Colab Pro for A100/V100 GPU")
        print("   - Using a smaller model")
        print("   - Enabling gradient checkpointing")
    else:
        print("\nSufficient GPU memory for Llama-3.1-8B")
else:
    print("No GPU available!")
    print("   Go to Runtime -> Change runtime type -> GPU")
    print("   This notebook requires a GPU to run.")

In [ ]:
# HuggingFace authentication (required for Llama-3.1-8B)
from huggingface_hub import notebook_login

print("Login to HuggingFace to access Llama-3.1-8B")
print("   Using model: Qwen/Qwen3-8B")
print("   Then enter your HF token below:\n")

notebook_login()

## 2. Get the Code

Clone the ZIP-RC repository or upload the required files. We need:
- `zip_rc_model.py` - Core model with ZIP tokens
- `generate_rollouts.py` - Rollout generation script
- `score_rollouts.py` - Scoring script
- `build_prefix_dataset.py` - Dataset builder

In [ ]:
# Option A: Clone from GitHub (if available)
# !git clone https://github.com/your-repo/zip-rc.git
# %cd zip-rc

# Option B: Upload files manually
# Use the file browser on the left to upload:
#   - zip_rc_model.py
#   - generate_rollouts.py
#   - score_rollouts.py
#   - build_prefix_dataset.py

# Option C: Create files inline (we'll do this below)
print("We'll create the necessary files inline in this notebook.")
print("   Alternatively, upload the .py files from the repo.")

### Create Core Files Inline

Let's create the ZIP-RC model and helper functions directly in this notebook.

In [ ]:
%%writefile zip_rc_model.py
"""
ZIP-RC Model Wrapper

Adds reserved ZIP tokens to a causal LM for joint (reward, length) prediction.
Each ZIP token maps to one cell in a BV x BT grid:
  - BV = number of reward bins (default 2: wrong/correct)
  - BT = number of length bins (default 5: logarithmic remaining-length buckets)
  - Total ZIP tokens = BV * BT
"""

import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer

# Default bin configuration
DEFAULT_BV = 2   # reward bins: 0=wrong, 1=correct
DEFAULT_BT = 5   # length bins: 0-9, 10-19, 20-39, 40-79, 80+

# Length bin boundaries (upper bounds, inclusive)
LENGTH_BIN_EDGES = [9, 19, 39, 79]  # bin 0: 0-9, bin 1: 10-19, bin 2: 20-39, bin 3: 40-79, bin 4: 80+


def tokens_left_to_bin(tokens_left: int) -> int:
    """Map remaining token count to a length bin index."""
    for i, edge in enumerate(LENGTH_BIN_EDGES):
        if tokens_left <= edge:
            return i
    return len(LENGTH_BIN_EDGES)  # last open-ended bin


def joint_label(reward_bin: int, length_bin: int, bt: int = DEFAULT_BT) -> int:
    """Flatten (reward_bin, length_bin) into a single class index."""
    return reward_bin * bt + length_bin


class ZipRCOutput:
    __slots__ = ("token_logits", "generation_logits", "zip_logits")

    def __init__(self, token_logits, generation_logits, zip_logits):
        self.token_logits = token_logits
        self.generation_logits = generation_logits
        self.zip_logits = zip_logits


class ZipRCModel(nn.Module):
    """Causal LM with reserved ZIP tokens for joint (reward, length) prediction."""

    def __init__(
        self,
        model_name_or_path: str = "Qwen/Qwen3-8B",
        bv: int = DEFAULT_BV,
        bt: int = DEFAULT_BT,
        freeze_backbone: bool = True,
        load_in_8bit: bool = False,
    ):
        super().__init__()
        self.bv = bv
        self.bt = bt
        self.num_zip_tokens = bv * bt

        self.tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
        
        # Load model with optional 8-bit quantization for memory efficiency
        load_kwargs = {"torch_dtype": torch.float16}
        if load_in_8bit:
            load_kwargs["load_in_8bit"] = True
            load_kwargs["device_map"] = "auto"
        
        self.model = AutoModelForCausalLM.from_pretrained(model_name_or_path, **load_kwargs)
        self.original_vocab_size = len(self.tokenizer)

        # Reserve ZIP tokens
        zip_tokens = [f"<ZIP_{i}>" for i in range(self.num_zip_tokens)]
        self.tokenizer.add_tokens(zip_tokens)
        self.model.resize_token_embeddings(len(self.tokenizer))

        if freeze_backbone:
            for param in self.model.parameters():
                param.requires_grad = False
            for param in self.model.lm_head.parameters():
                param.requires_grad = True

    def forward(self, input_ids, attention_mask=None, **kwargs):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **kwargs,
        )
        logits = outputs.logits  # (B, T, V + num_zip_tokens)

        # Split logits
        normal_logits = logits[..., :self.original_vocab_size]
        zip_logits = logits[..., -self.num_zip_tokens:]  # (B, T, BV*BT)

        # Mask ZIP tokens for generation
        generation_logits = logits.clone()
        generation_logits[..., -self.num_zip_tokens:] = float("-inf")

        return ZipRCOutput(
            token_logits=normal_logits,
            generation_logits=generation_logits,
            zip_logits=zip_logits,
        )

    def predict_zip(self, input_ids, attention_mask=None):
        """Return marginal reward and length distributions from the joint ZIP logits."""
        out = self.forward(input_ids, attention_mask)
        joint_probs = torch.softmax(out.zip_logits[:, -1, :], dim=-1)  # (B, BV*BT)
        joint_grid = joint_probs.view(-1, self.bv, self.bt)  # (B, BV, BT)

        # Marginals
        reward_probs = joint_grid.sum(dim=-1)  # (B, BV)
        length_probs = joint_grid.sum(dim=-2)  # (B, BT)

        # Expected values
        reward_vals = torch.arange(self.bv, device=joint_probs.device, dtype=torch.float)
        length_vals = torch.arange(self.bt, device=joint_probs.device, dtype=torch.float)

        expected_reward = (reward_probs * reward_vals).sum(dim=-1)
        expected_length = (length_probs * length_vals).sum(dim=-1)

        return expected_reward, expected_length

print("zip_rc_model.py created!")

## 3. Create Sample Data

Let's create a small set of math problems for our pipeline demo.

In [ ]:
import os
import json

# Create data directory
os.makedirs("data", exist_ok=True)

# Sample math problems (simple arithmetic)
sample_prompts = [
    {"prompt": "What is 15 + 27?", "answer": "42"},
    {"prompt": "What is 8 * 7?", "answer": "56"},
    {"prompt": "What is 100 - 37?", "answer": "63"},
    {"prompt": "What is 144 / 12?", "answer": "12"},
    {"prompt": "What is 25 + 75?", "answer": "100"},
    {"prompt": "What is 9 * 9?", "answer": "81"},
    {"prompt": "What is 200 - 88?", "answer": "112"},
    {"prompt": "What is 64 / 8?", "answer": "8"},
    {"prompt": "What is 33 + 67?", "answer": "100"},
    {"prompt": "What is 12 * 11?", "answer": "132"},
]

# Write to JSONL file
with open("data/prompts.jsonl", "w") as f:
    for item in sample_prompts:
        f.write(json.dumps(item) + "\n")

print(f"Created data/prompts.jsonl with {len(sample_prompts)} math problems")
print("\nSample problems:")
for item in sample_prompts[:3]:
    print(f"  Q: {item['prompt']} -> A: {item['answer']}")

## 4. Step 1: Generate Rollouts

Generate completions from the base Llama-3.1-8B model.

**What this does:**
- Loads the base LLM (no ZIP tokens yet)
- Generates one completion per prompt
- Saves completions + token IDs for later processing

In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm

MODEL_NAME = "Qwen/Qwen3-8B"
PROMPTS_PATH = "data/prompts.jsonl"
OUTPUT_PATH = "data/rollouts.jsonl"
MAX_NEW_TOKENS = 64  # Shorter for demo

print("Loading model (this may take a few minutes)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, 
    torch_dtype=torch.float16,
    device_map="auto"  # Automatic device placement
)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded on {model.device}")

# Load prompts
with open(PROMPTS_PATH) as f:
    prompts = [json.loads(line) for line in f]

print(f"\nGenerating {len(prompts)} rollouts...")

results = []
for row in tqdm(prompts):
    # Format as instruction
    text = f"Q: {row['prompt']}\nA:"
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,  # greedy for reproducibility
            pad_token_id=tokenizer.pad_token_id,
        )

    completion_ids = output_ids[0, prompt_len:].tolist()
    completion = tokenizer.decode(completion_ids, skip_special_tokens=True)

    results.append({
        "prompt": row["prompt"],
        "answer": row["answer"],
        "completion": completion.strip(),
        "completion_ids": completion_ids,
    })

# Save results
with open(OUTPUT_PATH, "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")

print(f"\nGenerated {len(results)} rollouts")
print(f"   Saved to {OUTPUT_PATH}")

# Show examples
print("\nSample completions:")
for r in results[:3]:
    print(f"  Q: {r['prompt']}")
    print(f"  A: {r['completion'][:100]}..." if len(r['completion']) > 100 else f"  A: {r['completion']}")
    print()

In [ ]:
# Free up GPU memory
del model
torch.cuda.empty_cache()
print("Cleared GPU memory")

## 5. Step 2: Score Rollouts

Evaluate each completion for correctness by extracting the final number and comparing to ground truth.

**Scoring logic:**
- Extract the last number from the completion
- Compare to expected answer
- reward = 1 if correct, 0 if wrong

In [ ]:
import json
import re

INPUT_PATH = "data/rollouts.jsonl"
OUTPUT_PATH = "data/scored_rollouts.jsonl"


def extract_number(text: str) -> str | None:
    """Extract the last number from text."""
    matches = re.findall(r"-?\d+\.?\d*", text)
    return matches[-1] if matches else None


def normalize_answer(ans: str) -> str:
    """Strip whitespace and trailing .0 for comparison."""
    ans = ans.strip()
    if ans.endswith(".0"):
        ans = ans[:-2]
    return ans


# Load rollouts
with open(INPUT_PATH) as f:
    rollouts = [json.loads(line) for line in f]

# Score each rollout
correct = 0
results = []
for row in rollouts:
    predicted = extract_number(row["completion"])
    expected = normalize_answer(row["answer"])

    if predicted is not None:
        predicted = normalize_answer(predicted)

    reward = 1 if predicted == expected else 0
    correct += reward

    results.append({**row, "predicted_answer": predicted, "reward": reward})

# Save scored rollouts
with open(OUTPUT_PATH, "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")

print(f"Scored {len(results)} rollouts")
print(f"   Correct: {correct}/{len(results)} ({100*correct/len(results):.1f}%)")
print(f"   Saved to {OUTPUT_PATH}")

# Show details
print("\nScoring details:")
for r in results[:5]:
    status = "CORRECT" if r['reward'] == 1 else "WRONG"
    print(f"  [{status}] Q: {r['prompt']} | Expected: {r['answer']} | Got: {r['predicted_answer']}")

## 6. Step 3: Build Prefix Dataset

Create training examples for ZIP-RC. For each completion with T tokens, we create T training examples—one per prefix length.

**For each prefix:**
- Calculate `tokens_left = T - t`
- Map to `length_bin` (0-4)
- Compute `joint_label = reward_bin * 5 + length_bin`

In [ ]:
import json
from collections import Counter
from transformers import AutoTokenizer

# Import helpers from our model file
from zip_rc_model import DEFAULT_BV, DEFAULT_BT, tokens_left_to_bin, joint_label

MODEL_NAME = "Qwen/Qwen3-8B"
INPUT_PATH = "data/scored_rollouts.jsonl"
OUTPUT_PATH = "data/prefix_dataset.jsonl"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

with open(INPUT_PATH) as f:
    rollouts = [json.loads(line) for line in f]

print(f"\nBuilding prefix dataset from {len(rollouts)} rollouts...")

examples = []
for row in rollouts:
    # Reconstruct full input: prompt + completion tokens
    prompt_text = f"Q: {row['prompt']}\nA:"
    prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    completion_ids = row["completion_ids"]
    T = len(completion_ids)

    if T == 0:
        continue

    reward_bin = row["reward"]  # already 0 or 1

    # Walk every prefix of the completion
    for t in range(1, T + 1):
        tokens_left = T - t
        length_bin = tokens_left_to_bin(tokens_left)
        label = joint_label(reward_bin, length_bin, bt=DEFAULT_BT)

        input_ids = prompt_ids + completion_ids[:t]
        attention_mask = [1] * len(input_ids)

        examples.append({
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "reward_bin": reward_bin,
            "length_bin": length_bin,
            "joint_label": label,
        })

# Save dataset
with open(OUTPUT_PATH, "w") as f:
    for ex in examples:
        f.write(json.dumps(ex) + "\n")

print(f"\nBuilt {len(examples)} prefix examples from {len(rollouts)} rollouts")
print(f"   Joint labels range: 0 to {DEFAULT_BV * DEFAULT_BT - 1}")
print(f"   Saved to {OUTPUT_PATH}")

# Show label distribution
labels = [ex["joint_label"] for ex in examples]
dist = Counter(labels)
print(f"\nLabel distribution:")
for label in sorted(dist.keys()):
    reward = label // DEFAULT_BT
    length = label % DEFAULT_BT
    print(f"   Label {label} (reward={reward}, length_bin={length}): {dist[label]} examples")

## 7. Step 4: Train ZIP-RC Model

Now we train the model to predict the correct ZIP token at each prefix position. We:
1. Load ZipRCModel (base LLM + 10 ZIP tokens)
2. Freeze backbone, train only LM head
3. Train with cross-entropy loss on joint labels

In [ ]:
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from zip_rc_model import ZipRCModel, DEFAULT_BV, DEFAULT_BT

# Training configuration
MODEL_NAME = "Qwen/Qwen3-8B"
DATASET_PATH = "data/prefix_dataset.jsonl"
OUTPUT_DIR = "zip_rc_checkpoint"
BATCH_SIZE = 2  # Small batch size for memory
NUM_EPOCHS = 3
LEARNING_RATE = 1e-4
MAX_SEQ_LEN = 256


class PrefixDataset(Dataset):
    """Dataset for ZIP-RC training."""
    
    def __init__(self, path: str, max_len: int = 256):
        with open(path) as f:
            self.examples = [json.loads(line) for line in f]
        self.max_len = max_len
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        ex = self.examples[idx]
        input_ids = ex["input_ids"][:self.max_len]
        attention_mask = ex["attention_mask"][:self.max_len]
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "joint_label": torch.tensor(ex["joint_label"], dtype=torch.long),
        }


def collate_fn(batch):
    """Pad sequences to same length."""
    max_len = max(len(item["input_ids"]) for item in batch)
    
    input_ids = []
    attention_mask = []
    labels = []
    
    for item in batch:
        pad_len = max_len - len(item["input_ids"])
        input_ids.append(torch.cat([item["input_ids"], torch.zeros(pad_len, dtype=torch.long)]))
        attention_mask.append(torch.cat([item["attention_mask"], torch.zeros(pad_len, dtype=torch.long)]))
        labels.append(item["joint_label"])
    
    return {
        "input_ids": torch.stack(input_ids),
        "attention_mask": torch.stack(attention_mask),
        "joint_label": torch.stack(labels),
    }


print("Loading dataset...")
dataset = PrefixDataset(DATASET_PATH, max_len=MAX_SEQ_LEN)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
print(f"   {len(dataset)} examples, {len(dataloader)} batches")

print("\nLoading ZIP-RC model...")
print("   (This may take a few minutes)")
model = ZipRCModel(MODEL_NAME, bv=DEFAULT_BV, bt=DEFAULT_BT, freeze_backbone=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# Count trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nParameters:")
print(f"   Total: {total_params:,}")
print(f"   Trainable: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")

# Setup training
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE
)
criterion = nn.CrossEntropyLoss()

print(f"\nStarting training for {NUM_EPOCHS} epochs...\n")

In [ ]:
# Training loop
model.train()

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(dataloader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}")
    for batch in pbar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["joint_label"].to(device)
        
        # Forward pass
        outputs = model(input_ids, attention_mask)
        
        # Get ZIP logits at last position for each sequence
        # Find actual last position (not padding)
        seq_lens = attention_mask.sum(dim=1) - 1  # 0-indexed
        zip_logits = outputs.zip_logits[torch.arange(len(seq_lens)), seq_lens]  # (B, 10)
        
        # Compute loss
        loss = criterion(zip_logits, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Track metrics
        epoch_loss += loss.item()
        preds = zip_logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += len(labels)
        
        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "acc": f"{100*correct/total:.1f}%"
        })
    
    avg_loss = epoch_loss / len(dataloader)
    accuracy = 100 * correct / total
    print(f"\nEpoch {epoch + 1} Summary:")
    print(f"  Average Loss: {avg_loss:.4f}")
    print(f"  Accuracy: {accuracy:.1f}%\n")

In [ ]:
# Save the trained model
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save model weights and tokenizer
model.model.save_pretrained(OUTPUT_DIR)
model.tokenizer.save_pretrained(OUTPUT_DIR)

# Save config
config = {
    "bv": model.bv,
    "bt": model.bt,
    "original_vocab_size": model.original_vocab_size,
    "num_zip_tokens": model.num_zip_tokens,
}
with open(f"{OUTPUT_DIR}/zip_rc_config.json", "w") as f:
    json.dump(config, f)

print(f"Model saved to {OUTPUT_DIR}/")
print(f"   Files: {os.listdir(OUTPUT_DIR)}")

## 8. Step 5: Test Inference

Let's test our trained ZIP-RC model! We'll see if it can predict:
- **Expected Reward**: Probability of getting the right answer (0-1)
- **Expected Length**: Which length bin the remaining tokens fall into (0-4)

In [ ]:
import torch
from zip_rc_model import ZipRCModel, LENGTH_BIN_EDGES

# Length bin descriptions
LENGTH_BIN_NAMES = ["0-9 tokens", "10-19 tokens", "20-39 tokens", "40-79 tokens", "80+ tokens"]

def test_introspection(model, tokenizer, prompt: str, partial_answer: str = ""):
    """Test ZIP-RC introspection on a prompt + partial answer."""
    # Format input
    text = f"Q: {prompt}\nA:{partial_answer}"
    inputs = tokenizer(text, return_tensors="pt").to(model.model.device)
    
    # Get predictions
    with torch.no_grad():
        expected_reward, expected_length = model.predict_zip(**inputs)
    
    reward = expected_reward.item()
    length_bin = int(round(expected_length.item()))
    length_bin = max(0, min(length_bin, 4))  # Clamp to valid range
    
    print(f"Input: '{text}'")
    print(f"  Expected Reward: {reward:.3f} ({'likely correct' if reward > 0.5 else 'likely wrong'})")
    print(f"  Expected Length Bin: {length_bin} ({LENGTH_BIN_NAMES[length_bin]})")
    print()
    
    return reward, length_bin


print("Testing ZIP-RC Introspection\n")
print("="*60)

# Test cases
test_cases = [
    # Just the question
    ("What is 15 + 27?", ""),
    # Partial correct answer
    ("What is 15 + 27?", " 4"),
    # Full correct answer
    ("What is 15 + 27?", " 42"),
    # Wrong direction
    ("What is 15 + 27?", " 3"),
    # New problem
    ("What is 8 * 7?", ""),
    ("What is 8 * 7?", " 5"),
    ("What is 8 * 7?", " 56"),
]

tokenizer = model.tokenizer

for prompt, partial in test_cases:
    test_introspection(model, tokenizer, prompt, partial)

## 9. Save Checkpoint to Google Drive (Optional)

Download your trained model to Google Drive for later use.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy checkpoint to Drive
import shutil

DRIVE_PATH = "/content/drive/MyDrive/zip_rc_checkpoint"
shutil.copytree(OUTPUT_DIR, DRIVE_PATH, dirs_exist_ok=True)

print(f"Checkpoint saved to Google Drive: {DRIVE_PATH}")

## Summary

Congratulations! You've successfully run the complete ZIP-RC pipeline:

1. **Setup** - Installed dependencies and authenticated with HuggingFace
2. **Generate Rollouts** - Created completions from Llama-3.1-8B
3. **Score Rollouts** - Evaluated correctness of each completion
4. **Build Dataset** - Created prefix training examples with joint labels
5. **Train ZIP-RC** - Fine-tuned the model to predict (reward, length)
6. **Test Inference** - Verified the model can introspect during generation

### Key Insights

- **Zero Overhead**: The ZIP tokens are computed in the same forward pass as text tokens
- **Joint Prediction**: The model predicts both reward (will I succeed?) and length (how many tokens left?)
- **Adaptive Inference**: This enables beam pruning, early stopping, and other test-time optimizations

### Next Steps

- Train on more diverse prompts and longer completions
- Use the trained model for Best-of-N selection
- Implement beam search with ZIP-guided pruning
- Experiment with different binning strategies